<a name="top"></a><img src="../images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 4.3：常见 Pass 惯用法

**上一节：[FIRRTL AST 遍历](4.2_firrtl_ast_traversal.ipynb)**<br>
**下一节：[FIRRTL 变换示例](4.4_firrtl_add_ops_per_module.ipynb)**

### 添加语句
假设我们要编写一个 pass，将嵌套的 DoPrim 表达式拆分，从而将以下代码：
```
circuit Top:
  module Top :
  input x: UInt<3>
  input y: UInt<3>
  input z: UInt<3>
  output o: UInt<3>
  o <= add(x, add(y, z))
```
转换为：
```
circuit Top:
  module Top :
  input x: UInt<3>
  input y: UInt<3>
  input z: UInt<3>
  output o: UInt<3>
  node GEN_1 = add(y, z)
  o <= add(x, GEN_1)
```

我们首先需要遍历 AST 到每个 Statement 和 Expression。然后，当我们看到 DoPrim 时，我们需要在模块的 body 中添加一个新的 DefNode，并插入一个指向该 DefNode 的引用来替换 DoPrim。下面的代码实现了这一点（并保留了 Info 标记）。注意，`Namespace` 是一个在 [Namespace.scala](https://github.com/ucb-bar/firrtl/blob/master/src/main/scala/firrtl/Namespace.scala) 中的工具函数。

```scala
object Splitter extends Pass {
  def name = "Splitter!"
  /** 在每个模块上运行 splitM **/
  def run(c: Circuit): Circuit = c.copy(modules = c.modules map(splitM(_)))

  /** 在每个模块的 body 上运行 splitS **/
  def splitM(m: DefModule): DefModule = m map splitS(Namespace(m))

  /** 在所有子 Expression 上运行 splitE。
    * 如果 stmts 包含额外的语句，返回包含这些语句和新语句的 Block；
    * 否则，返回新语句。 */
  def splitS(namespace: Namespace)(s: Statement): Statement = {
    val block = mutable.ArrayBuffer[Statement]()
    s match {
      case s: HasInfo => 
        val newStmt = s map splitE(block, namespace, s.info)
        block.length match {
          case 0 => newStmt
          case _ => Block(block.toSeq :+ newStmt)
        }
      case s => s map splitS(namespace)
  }

  /** 在所有子表达式上运行 splitE。
    * 如果 e 是 DoPrim，向 block 添加新的 DefNode 并返回指向
    * 该 DefNode 的引用；否则返回 e。*/
  def splitE(block: mutable.ArrayBuffer[Statement], namespace: Namespace, 
             info: Info)(e: Expression): Expression = e map splitE(block, namespace, info) match {
    case e: DoPrim =>
      val newName = namespace.newTemp
      block += DefNode(info, newName, e)
      Ref(newName, e.tpe)
    case _ => e
  }
}
```
### 删除语句
假设我们要编写一个 pass，内联所有值为字面量的 DefNode，从而将以下代码：
```
circuit Top:
  module Top :
  input x: UInt<3>
  output o: UInt<4>
  node y = UInt(1)
  o <= add(x, y)
```
转换为：
```
circuit Top:
  module Top :
  input x: UInt<3>
  output y: UInt<4>
  o <= add(x, UInt(1))
```

我们首先需要遍历 AST 到每个 Statement 和 Expression。然后，当我们看到一个指向 Literal 的 DefNode 时，需要将其存储到哈希映射中，并返回一个 EmptyStmt（从而删除该 DefNode）。然后，每当我们看到对已删除 DefNode 的引用时，必须插入对应的 Literal。

```scala
object Inliner extends Pass {
  def name = "Inliner!"
  /** 在每个模块上运行 inlineM **/
  def run(c: Circuit): Circuit = c.copy(modules = c.modules map(inlineM(_)))

  /** 在每个模块的 body 上运行 inlineS **/
  def inlineM(m: DefModule): DefModule = m map inlineS(mutable.HashMap[String, Expression]())

  /** 在所有子 Expression 上运行 inlineE，然后在子语句上运行 inlineS。
    * 如果语句是包含字面量的 DefNode，更新 values 并
    * 返回 EmptyStmt；否则返回原语句。 */
  def inlineS(values: mutable.HashMap[String, Expression])(s: Statement): Statement =
    s map inlineE(values) map inlineS(values) match {
      case d: DefNode => d.value match {
        case l: Literal =>
          values(d.name) = l
          EmptyStmt
        case _ => d
      }
      case o => o 
    }

  /** 如果 e 是一个名称包含在 values 中的引用，
    *  返回 values(e.name)；否则在所有子表达式上运行 inlineE。*/
  def inlineE(values: mutable.HashMap[String, Expression])(e: Expression): Expression = e match {
    case e: Ref if values.contains(e.name) => values(e.name)
    case _ => e map inlineE(values)
  }
}
```

### 添加一个 PrimOp
这个有用吗？通过向 [firrtl 仓库](https://github.com/freechipsproject/firrtl) 提交 issue 来告诉 [@azidar](https://github.com/azidar)！

### 替换语句
这个有用吗？通过向 [firrtl 仓库](https://github.com/freechipsproject/firrtl) 提交 issue 来告诉 [@azidar](https://github.com/azidar)！